# Part 2: Boosting, Practical Tabular ML, and Projects

This continues the course. Prerequisite: completion of Part 1 (metrics, CV, Pipeline, trees, and forests).

## Learning roadmap: what to study next and in what order

| Roadmap stage | Topic | Purpose | Estimated time |
|---|---|---|---|
| **A** | Feature engineering | Can improve performance more than switching models | 1–2 days |
| **B** | Gradient Boosting in depth (XGBoost / LightGBM / CatBoost) | A standard approach for tabular data | 2–3 days |
| **C** | Interpretation, calibration, thresholds, ensembles | Explain predictions and make model-based decisions | 2 days |
| **D** | Time series validation and leakage in real tasks | Reliable evaluation on temporal data | 1 day |
| **E** | **Projects** (Section G of this notebook) | Reinforce the material through practice | 1–2 weeks |
| **F** | Part 3: PyTorch and Deep Learning | Images, text, and sequences | 2–4 weeks |
| **G** | NLP/Transformers, MLOps (deployment, monitoring) | Practical application | later |

Recommended order: **complete Part 2 before Part 3.** Boosting often outperforms neural networks on tabular data. Understanding validation, leakage, and metrics helps evaluate DL results reliably.

## Contents
- **0.** The dataset used throughout this part
- **A.** Feature engineering
- **B.** Gradient Boosting in depth
- **C.** Interpretation, calibration, thresholds, ensembles
- **D.** Time: a subtle source of leakage
- **E.** Final evaluation: once on test
- **F.** Saving models and inference
- **G.** Projects


In [ ]:
# Install missing dependencies: pip install xgboost lightgbm catboost
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 50)

import sklearn, xgboost, lightgbm
print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__)
try:
    import catboost; print("catboost", catboost.__version__)
except ImportError:
    catboost = None; print("catboost is not installed: pip install catboost (its example will be skipped)")

## 0. The dataset used throughout this part

Synthetic but challenging: nonlinearities, feature interactions, missing values, categories, and dates. The task is to predict customer attrition (`churn`). The true relationships are **deliberately** embedded in interactions and dates so that feature engineering has meaningful patterns to uncover.


In [ ]:
rng = np.random.RandomState(7)
n = 12000
df = pd.DataFrame({
    "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.randint(0, 700, n), unit="D"),
    "age": rng.randint(18, 75, n).astype(float),
    "income": rng.lognormal(8.2, 0.7, n),
    "city": rng.choice(["Lviv", "Kyiv", "Odesa", "Kharkiv", "Dnipro", "Ternopil"], n),
    "plan": rng.choice(["basic", "plus", "pro"], n, p=[.5, .3, .2]),
    "tickets": rng.poisson(2, n),
    "logins_30d": rng.poisson(12, n),
    "spend_90d": rng.gamma(2.0, 60, n),
})
tenure = (pd.Timestamp("2025-01-01") - df.signup_date).dt.days
logit = (
    -1.2
    + 0.35 * df.tickets
    - 0.08 * df.logins_30d
    + 0.9 * (df.spend_90d / (df.income / 12) > 0.08)     # spending relative to income (interaction!)
    + 0.9 * ((tenure < 120) & (df.plan == "basic"))        # new customers on the basic plan
    + 0.6 * (df.city == "Kyiv") * (df.age < 30)
    - 0.004 * (tenure - 300)
)
df["churn"] = (rng.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)
df.loc[rng.rand(n) < .08, "age"] = np.nan
df.loc[rng.rand(n) < .05, "city"] = "missing"
print(df.shape, "| churn rate:", df.churn.mean().round(3))
df.head()

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

# IMPORTANT: reserve test from the start; do not use it until the end of this part
df_dev, df_test = train_test_split(df, test_size=0.2, random_state=0, stratify=df.churn)
df_dev = df_dev.reset_index(drop=True); df_test = df_test.reset_index(drop=True)
print(df_dev.shape, df_test.shape)

## A. Feature engineering

A model uses only the features provided to it. Trees do not always efficiently reconstruct **ratios**, **differences**, **aggregations**, or **date components**. Domain knowledge helps create these features manually.

Main techniques:
1. **Dates** → year, month, day of week, days since an event (tenure), weekend/weekday.
2. **Ratios and differences** between columns (`spend / income`).
3. **Log transformations** for skewed distributions (income, prices). Less important for trees, but often important for linear models and neural networks.
4. **Group aggregations**: mean/median/count per group (`mean income by city`). ⚠️ Compute them **only on train** to avoid leakage.
5. **Bins** (age → age groups), missingness flags (`age_missing`).
6. **Target encoding** of categories (below), also with leakage protection.


In [ ]:
REF_DATE = pd.Timestamp("2025-01-01")

def add_features(d, city_stats=None):
    d = d.copy()
    d["tenure_days"] = (REF_DATE - d["signup_date"]).dt.days
    d["signup_month"] = d["signup_date"].dt.month
    d["signup_dow"] = d["signup_date"].dt.dayofweek
    d["spend_to_income"] = d["spend_90d"] / (d["income"] / 12)
    d["log_income"] = np.log1p(d["income"])
    d["tickets_per_login"] = d["tickets"] / (d["logins_30d"] + 1)
    d["age_missing"] = d["age"].isna().astype(int)
    d["is_new_basic"] = ((d["tenure_days"] < 120) & (d["plan"] == "basic")).astype(int)
    # group aggregation: compute on train, apply to all splits
    if city_stats is None:
        city_stats = d.groupby("city")["income"].median().rename("city_median_income")
    d = d.merge(city_stats, left_on="city", right_index=True, how="left")
    d["income_vs_city"] = d["income"] / d["city_median_income"]
    return d.drop(columns=["signup_date"]), city_stats

dev_fe, city_stats = add_features(df_dev)            # stats from train
test_fe, _ = add_features(df_test, city_stats)       # apply the same stats to test
dev_fe.head(3)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedKFold(5, shuffle=True, random_state=42)
cat_cols = ["city", "plan"]

def prep_native(d):
    d = d.copy()
    for c in cat_cols: d[c] = d[c].astype("category")
    return d

raw = prep_native(df_dev.drop(columns=["signup_date", "churn"]))
fe = prep_native(dev_fe.drop(columns=["churn"]))
y_dev = df_dev["churn"].values

for name, Xm in {"raw features": raw, "with feature engineering": fe}.items():
    m = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=0)
    s = cross_val_score(m, Xm, y_dev, cv=cv, scoring="roc_auc")
    print(f"{name:24s} AUC = {s.mean():.4f} ± {s.std():.4f}")

This is why FE can matter more than tuning: the same model, supplied with more informative features.

### Target encoding (and its leakage risk)
Instead of One-Hot encoding a feature with hundreds of categories, replace each category with its **mean target**. Computing this on all training data at once lets each row "see" its own target, causing leakage. Use **out-of-fold** encoding. In `sklearn >= 1.3`, `TargetEncoder` handles this with internal cross-fitting in `fit_transform`.


In [ ]:
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

num_cols = [c for c in dev_fe.columns if c not in cat_cols + ["churn"]]
pre = ColumnTransformer([
    ("te", TargetEncoder(target_type="binary", random_state=0), cat_cols),
    ("num", make_pipeline(__import__("sklearn.impute", fromlist=["SimpleImputer"]).SimpleImputer(strategy="median"), StandardScaler()), num_cols),
])
pipe = make_pipeline(pre, LogisticRegression(max_iter=2000))
s = cross_val_score(pipe, dev_fe.drop(columns="churn"), y_dev, cv=cv, scoring="roc_auc")
print(f"LogReg + TargetEncoder  AUC = {s.mean():.4f} ± {s.std():.4f}")

### 📝 Exercise A.1
Suggest 3 more features for this dataset (e.g., `is_weekend_signup`, `age_bucket`, `logins_per_ticket`), add them to `add_features`, and check via CV whether AUC improves. Which feature helped most? Which did not help?


In [ ]:
# Your solution here

### 📝 Exercise A.2
Deliberately introduce leakage: compute `mean churn by city` using ALL of df_dev and add it as a feature. Inspect CV AUC, then compare with `TargetEncoder`. Why does the first approach inflate the result, especially for rare categories?


In [ ]:
# Your solution here

## B. Gradient Boosting in depth

### The intuition
Boosting builds a model **additively**:

```text
F0(x) = mean y
F1(x) = F0(x) + lr · tree1(x)     # tree1 learns the residuals y − F0
F2(x) = F1(x) + lr · tree2(x)     # tree2 learns the residuals y − F1
...
```
For squared error, residuals are proportional to the negative loss gradient. For other losses (log loss, etc.), each tree learns the **negative gradient** of that loss, hence *gradient* boosting. This is the gradient descent from Part 1, but **in function space**, rather than weight space.

We will implement a minimal version manually.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

class SimpleGBR:
    def __init__(self, n_estimators=100, lr=0.1, max_depth=2):
        self.n, self.lr, self.depth = n_estimators, lr, max_depth
    def fit(self, X, y):
        self.f0 = y.mean()
        pred = np.full(len(y), self.f0)
        self.trees = []
        for _ in range(self.n):
            resid = y - pred                         # residuals are proportional to the negative MSE gradient
            t = DecisionTreeRegressor(max_depth=self.depth).fit(X, resid)
            pred += self.lr * t.predict(X)
            self.trees.append(t)
        return self
    def predict(self, X, n_trees=None):
        pred = np.full(len(X), self.f0)
        for t in self.trees[:n_trees]:
            pred += self.lr * t.predict(X)
        return pred

# one-dimensional nonlinear example for visualization
r = np.random.RandomState(0)
xs = np.sort(r.uniform(0, 10, 200))[:, None]
ys = np.sin(xs[:, 0]) + 0.3 * r.randn(200)
grid = np.linspace(0, 10, 400)[:, None]

fig, ax = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
gb = SimpleGBR(n_estimators=200, lr=0.1, max_depth=2).fit(xs, ys)
for a_, k in zip(ax, [1, 5, 30, 200]):
    a_.scatter(xs, ys, s=8, alpha=.5)
    a_.plot(grid, gb.predict(grid, n_trees=k), "r"); a_.set_title(f"{k} trees")
plt.show()

In the plot, 1 tree gives a coarse step function, while 200 trees give a smoother curve. Each new tree corrects earlier predictions.

### Key hyperparameters (similar concepts across libraries)

| Parameter | Effect | Typical values |
|---|---|---|
| `n_estimators` (`iterations`) | number of trees | many (1000+) **with early stopping** |
| `learning_rate` | each tree's contribution; lower lr requires more trees and often improves performance | 0.01–0.1 |
| `max_depth` / `num_leaves` | tree complexity; boosting often uses **shallow** trees | 3–8 (LGBM: num_leaves 15–63) |
| `min_child_samples` / `min_child_weight` | minimum leaf support; helps control overfitting | 20–100 |
| `subsample` | fraction of rows per tree (stochastic boosting) | 0.6–1.0 |
| `colsample_bytree` | fraction of features per tree | 0.5–1.0 |
| `reg_lambda`, `reg_alpha` | L2/L1 penalties on leaf weights | 0–10 |

### Early stopping: a useful practical technique
Set a large tree budget and supply a validation set. Training stops after N rounds without validation improvement. This **selects the number of trees automatically**. ⚠️ Early-stopping validation is not test; reserve a separate portion of train.

### XGBoost vs LightGBM vs CatBoost
- **XGBoost**: a mature, robust option, commonly growing trees level by level (level-wise).
- **LightGBM**: designed for fast training, growing leaf by leaf (leaf-wise); effective on large datasets but can overfit small ones (control `num_leaves` and `min_child_samples`).
- **CatBoost**: strong native **categorical** handling (ordered target encoding), symmetric trees, and useful defaults.
- **HistGradientBoosting (sklearn)**: a useful baseline without extra dependencies.


In [ ]:
import lightgbm as lgb
import xgboost as xgb

# Separate validation split within dev for early stopping
X_all = prep_native(dev_fe.drop(columns=["churn"]))
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_dev, test_size=0.2, random_state=1, stratify=y_dev)

# --- XGBoost
xgbm = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_lambda=2.0, tree_method="hist", enable_categorical=True,
    eval_metric="auc", early_stopping_rounds=100, random_state=0,
)
xgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
p = xgbm.predict_proba(X_val)[:, 1]
print(f"XGBoost   best_iter={xgbm.best_iteration:4d}  val AUC={roc_auc_score(y_val, p):.4f}")

# --- LightGBM
lgbm = lgb.LGBMClassifier(
    n_estimators=3000, learning_rate=0.03, num_leaves=15, min_child_samples=40,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0, random_state=0, verbose=-1,
)
lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
         callbacks=[lgb.early_stopping(100, verbose=False)])
p = lgbm.predict_proba(X_val)[:, 1]
print(f"LightGBM  best_iter={lgbm.best_iteration_:4d}  val AUC={roc_auc_score(y_val, p):.4f}")

# --- sklearn HistGB
hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=4, max_iter=2000, early_stopping=True,
                                     n_iter_no_change=50, validation_fraction=0.15,
                                     categorical_features="from_dtype", random_state=0)
hgb.fit(X_tr, y_tr)
p = hgb.predict_proba(X_val)[:, 1]
print(f"HistGB    best_iter={hgb.n_iter_:4d}  val AUC={roc_auc_score(y_val, p):.4f}")

In [ ]:
# --- CatBoost (pass categories as strings, without encoding)
if catboost is not None:
    from catboost import CatBoostClassifier
    Xc_tr = X_tr.copy(); Xc_val = X_val.copy()
    for c in cat_cols:
        Xc_tr[c] = Xc_tr[c].astype(str); Xc_val[c] = Xc_val[c].astype(str)
    cb = CatBoostClassifier(iterations=3000, learning_rate=0.05, depth=5, eval_metric="AUC",
                            early_stopping_rounds=100, random_seed=0, verbose=0)
    cb.fit(Xc_tr, y_tr, cat_features=cat_cols, eval_set=(Xc_val, y_val))
    p = cb.predict_proba(Xc_val)[:, 1]
    print(f"CatBoost  best_iter={cb.get_best_iteration():4d}  val AUC={roc_auc_score(y_val, p):.4f}")

In [ ]:
# Learning curve: train vs val over iterations (LightGBM)
lg = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=15, min_child_samples=40, verbose=-1, random_state=0)
lg.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], eval_metric="auc")
ev = lg.evals_result_
plt.plot(ev["training"]["auc"], label="train"); plt.plot(ev["valid_1"]["auc"], label="val")
plt.axvline(int(np.argmax(ev["valid_1"]["auc"])), color="gray", ls="--", label="best val")
plt.xlabel("iteration"); plt.ylabel("AUC"); plt.legend(); plt.title("Train improves, val plateaus and drops = overfitting"); plt.show()

### 📝 Exercise B.1
Explore `learning_rate`: for LightGBM, try lr ∈ {0.3, 0.1, 0.03, 0.01} with early stopping. Record best_iteration and validation AUC for each. How are lr and the number of trees related? Does a smaller lr always improve AUC?


In [ ]:
# Your solution here

### 📝 Exercise B.2
Explore complexity: vary LightGBM's `num_leaves` ∈ {4, 8, 15, 31, 63, 127}, keeping everything else fixed. Plot validation AUC. Where does overfitting begin? Repeat for XGBoost with `max_depth` 2..10.


In [ ]:
# Your solution here

### 📝 Exercise B.3
Tune LightGBM using `RandomizedSearchCV` (the `scikit-learn` API; `n_estimators=400`, no early stopping during the search) with 5-fold CV on dev. Then train the best configuration with early stopping. Compare with the defaults.


In [ ]:
# Your solution here

### 📝 Exercise B.4
Implement **subsampling** in `SimpleGBR` (each tree uses a random 70% of rows) and compare it with the full-data version on the sine-wave example. Why can this reduce overfitting?


In [ ]:
# Your solution here

## C. Interpretation, calibration, thresholds, ensembles

### C1. Understanding what a model learned
- **Permutation importance** (on validation): covered in Part 1.
- **Partial Dependence (PDP)**: how the average prediction changes when one feature is varied while the other features remain as observed.
- **ICE**: similar to PDP, but plotted for individual examples to reveal variation.
- **SHAP** (`shap` library): decomposes **each** row's prediction into feature contributions. A widely used explanation method for boosting: `pip install shap`.


In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

fig, ax = plt.subplots(figsize=(12, 4))
X_pdp = X_val.copy()
X_pdp[["tenure_days", "tickets"]] = X_pdp[["tenure_days", "tickets"]].astype(float)   # cast integer features to float for PDP
PartialDependenceDisplay.from_estimator(
    lgbm, X_pdp, ["spend_to_income", "tenure_days", "tickets"], ax=ax, kind="average"
)
plt.tight_layout(); plt.show()

perm = permutation_importance(lgbm, X_val, y_val, scoring="roc_auc", n_repeats=5, random_state=0)
pd.Series(perm.importances_mean, index=X_val.columns).sort_values().tail(10).plot.barh(title="permutation importance (val)")
plt.show()

In [ ]:
# SHAP: optional
try:
    import shap
    explainer = shap.TreeExplainer(lgbm)
    sv = explainer.shap_values(X_val.iloc[:500])
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, X_val.iloc[:500], show=True)
except ImportError:
    print("shap is not installed: pip install shap, then rerun this cell")

### C2. Probability calibration
A model predicts "0.8", but are approximately 80 out of 100 such examples actually positive? Calibration is not needed for **ranking** (AUC), but matters for probability-based decisions (risk, pricing, expected value). Boosting and forests can produce distorted probabilities.

- **Reliability curve**: predicted probability on X, observed positive fraction on Y. Ideally follows the diagonal.
- **Brier score** (`brier_score_loss`) and **log loss** evaluate probability predictions.
- Calibration: `CalibratedClassifierCV(method="isotonic" / "sigmoid")`.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import brier_score_loss

base = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40, verbose=-1, random_state=0)
base.fit(X_tr, y_tr)
cal = CalibratedClassifierCV(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                             min_child_samples=40, verbose=-1, random_state=0), method="isotonic", cv=5)
cal.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(5, 5))
CalibrationDisplay.from_estimator(base, X_val, y_val, n_bins=10, ax=ax, name="raw")
CalibrationDisplay.from_estimator(cal, X_val, y_val, n_bins=10, ax=ax, name="isotonic")
plt.show()
for n_, m in {"raw": base, "calibrated": cal}.items():
    pp = m.predict_proba(X_val)[:, 1]
    print(f"{n_:11s} brier={brier_score_loss(y_val, pp):.4f}  logloss={log_loss(y_val, pp):.4f}  AUC={roc_auc_score(y_val, pp):.4f}")

### C3. Choosing a threshold by financial value rather than F1
In real applications, errors have different costs. Suppose a retention offer costs 10 UAH per customer receiving it, while losing a customer costs 100 UAH. Then:

- Offer a discount to someone who would stay anyway (FP): −10
- Offer it to someone who would leave and retain them (TP): +100 − 10 (assuming 100% success; in practice account for the probability of success)
- Do not offer it to someone who leaves (FN): 0 (the customer is lost, but this is the baseline outcome)
- Do not offer it to someone who stays (TN): 0

Maximize **profit**, rather than F1.


In [ ]:
pv = cal.predict_proba(X_val)[:, 1]
thrs = np.linspace(0.02, 0.9, 80)
profit = []
for t in thrs:
    pred = pv >= t
    tp = ((pred) & (y_val == 1)).sum(); fp = ((pred) & (y_val == 0)).sum()
    profit.append(tp * 90 - fp * 10)
best_t = thrs[int(np.argmax(profit))]
plt.plot(thrs, profit); plt.axvline(best_t, color="r", ls="--", label=f"optimal thr={best_t:.2f}")
plt.axvline(0.5, color="gray", ls=":", label="0.5"); plt.legend(); plt.xlabel("threshold"); plt.ylabel("profit"); plt.show()
print("optimal threshold:", round(best_t, 2), "| profit:", max(profit), "| at 0.5:", profit[int(np.argmin(abs(thrs-0.5)))])

### 📝 Exercise C.1
Change the costs (e.g., a 30 UAH discount and an 80 UAH loss) and observe how the optimal threshold shifts. Explain why a more expensive discount raises the threshold.


In [ ]:
# Your solution here

### 📝 Exercise C.2
Compare PDP and permutation importance. Find a feature that looks influential in one and less influential in the other. Why can this happen? Hint: correlated features.


In [ ]:
# Your solution here

### C4. Ensembles: Voting and Stacking
- **Voting**: averages predictions from different models. Works best when models are **diverse** and make different errors.
- **Stacking**: a meta-model learns to combine base-model predictions. Its training inputs must be **out-of-fold** predictions (sklearn handles this through `cv`).
- Improvements are often small (e.g., 0.2–1 percentage points of AUC), but may matter in competitions.


In [ ]:
from sklearn.ensemble import StackingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

Xn = dev_fe.drop(columns=["churn"]).copy()
oe = ColumnTransformer([("o", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)],
                       remainder="passthrough", verbose_feature_names_out=False)

def wrap(m): return make_pipeline(oe, SimpleImputer(strategy="median"), m)

lgb_m = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.05, num_leaves=15, min_child_samples=40, verbose=-1, random_state=0)
rf_m = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
lr_m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

models = {
    "lgbm": wrap(lgb_m),
    "rf": wrap(rf_m),
    "logreg": wrap(lr_m),
}
for k, m in models.items():
    s = cross_val_score(m, Xn, y_dev, cv=cv, scoring="roc_auc")
    print(f"{k:8s} AUC = {s.mean():.4f}")

stack = StackingClassifier(
    estimators=[(k, m) for k, m in models.items()],
    final_estimator=LogisticRegression(max_iter=1000), cv=5, stack_method="predict_proba", n_jobs=-1)
vote = VotingClassifier([(k, m) for k, m in models.items()], voting="soft")
for k, m in {"voting": vote, "stacking": stack}.items():
    s = cross_val_score(m, Xn, y_dev, cv=cv, scoring="roc_auc")
    print(f"{k:8s} AUC = {s.mean():.4f}")

### 📝 Exercise C.3
Add XGBoost and HistGB to the ensemble. Does AUC improve? Calculate the correlations between model predictions (out-of-fold using `cross_val_predict`). Why does combining two very similar models often add little value?


In [ ]:
# Your solution here

## D. Time: a subtle source of leakage

For data with a time component (orders, logs, prices), a **random** split/CV can train on the "future" and evaluate on the "past". Evaluation looks promising while deployed performance drops.

Guidelines:
1. Split by time: train = earlier, test = later.
2. Use `TimeSeriesSplit` (an expanding window) instead of KFold.
3. Build features from past information only: lags and rolling averages over **previous periods**, excluding the current target value.
4. Account for drift (distribution shift): conditions change, models become outdated, and monitoring is needed.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

t = np.arange(1500)
rs = np.random.RandomState(0)
y_ts = 0.02 * t + 5 * np.sin(2 * np.pi * t / 90) + np.cumsum(rs.randn(1500)) * 0.3 + rs.randn(1500)
ts = pd.DataFrame({"t": t, "y": y_ts})
ts["lag1"] = ts.y.shift(1); ts["lag7"] = ts.y.shift(7)
ts["roll7_past"] = ts.y.shift(1).rolling(7).mean()          # past values only!
ts = ts.dropna().reset_index(drop=True)
feats = ["t", "lag1", "lag7", "roll7_past"]

rf = RandomForestRegressor(200, min_samples_leaf=3, n_jobs=-1, random_state=0)
for name, splitter in {"KFold (shuffle): optimistic": KFold(5, shuffle=True, random_state=0),
                       "TimeSeriesSplit: time-aware": TimeSeriesSplit(5)}.items():
    r2 = cross_val_score(rf, ts[feats], ts["y"], cv=splitter, scoring="r2")
    print(f"{name:32s} R2 = {r2.mean():.3f}")

### 📝 Exercise D.1
Use only lag-based features by removing `t` and inspect how both estimates change. Then deliberately introduce leakage: `roll7_now = y.rolling(7).mean()` (including the current value). What happens to R²?


In [ ]:
# Your solution here

## E. Final evaluation: once on test

Now evaluate on the test set reserved at the beginning. **Once**, using the final model.


In [ ]:
final = lgb.LGBMClassifier(n_estimators=int(lgbm.best_iteration_ * 1.1), learning_rate=0.03, num_leaves=15, min_child_samples=40,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0, verbose=-1, random_state=0)
final.fit(X_all, y_dev)                                  # fit on ALL dev (train+val), tree count based on early stopping

X_test = prep_native(test_fe.drop(columns=["churn"]))
X_test["city"] = pd.Categorical(X_test["city"], categories=X_all["city"].cat.categories)
X_test["plan"] = pd.Categorical(X_test["plan"], categories=X_all["plan"].cat.categories)
pt = final.predict_proba(X_test)[:, 1]
print("TEST  AUC =", round(roc_auc_score(df_test.churn, pt), 4), "| PR-AUC =", round(average_precision_score(df_test.churn, pt), 4),
      "| logloss =", round(log_loss(df_test.churn, pt), 4))

## F. Saving models and inference

In production, preserve the **entire pipeline** (preprocessing + model) so it can be loaded together. Also save the expected columns and library versions.


In [ ]:
import joblib, json
joblib.dump({"model": final, "city_stats": city_stats, "cat_categories": {c: list(X_all[c].cat.categories) for c in cat_cols},
             "sklearn": sklearn.__version__, "lightgbm": lgb.__version__}, "churn_model.joblib")
bundle = joblib.load("churn_model.joblib")
print(type(bundle["model"]).__name__, "| city categories:", bundle["cat_categories"]["city"])

## G. Projects

Complete the projects in this order. Put each in a separate Git repository with a `README` describing the task, metric, results, and conclusions; an EDA notebook; a `src/` directory; and `requirements.txt`.

### Project 1: Titanic / House Prices (Kaggle), 2–3 days
Complete workflow: EDA → baseline → feature engineering → CV → boosting → submission. Focus on a **clean**, leakage-free process rather than the leaderboard result.

### Project 2: Telco Churn or Credit Default, 3–4 days
Imbalanced classes, calibration, cost-based thresholds, and SHAP explanations. Include business recommendations in the README.

### Project 3: Sales / demand forecasting (time series), 4–5 days
`TimeSeriesSplit`, lags, seasonality, and boosting compared with a naive baseline ("yesterday's value"). This demonstrates time-series modeling skills.

### Project 4: ML functionality for an online-store web application, 5–7 days
Combine ML with web development. Options include:
- **"Frequently bought together" recommendations** (item-item similarity based on shared purchases / cosine similarity), or
- **Product price / demand prediction**, or
- **Review classification** (positive/negative), or
- Serving a saved model through the application's HTTP API (`joblib.load` at startup, `predict` in the request handler), with caching.
Even with synthetic or collected educational data, this demonstrates the full path from data to model to product.

### Project 5: A Kaggle competition (any "Playground Series"), ongoing
New tabular tasks appear regularly. Practice Part 2: FE, boosting, ensembles, and reliable CV. Read discussions and leading notebooks after the competition.

### Readiness checklist for DL on tabular and other data
- [ ] Explain boosting vs bagging and the role of learning rate in boosting
- [ ] Use early stopping and explain why its validation set is not test
- [ ] Create date, ratio, and group-aggregation features without leakage
- [ ] Check calibration and choose thresholds based on error costs
- [ ] Know when to use KFold or TimeSeriesSplit
- [ ] Save and load an entire pipeline
- [ ] Complete at least 2 projects in Git repositories

**Next: Part 3 (PyTorch and Deep Learning).**
